# Machine Learning Pipeline: Belgian Motor Third-Party Liability Insurance Claims Prediction

## Project Overview

This notebook builds a **machine learning model to predict insurance claim severity** (the financial amount paid out) for Belgian motor third-party liability insurance claims.

### Why This Matters
Insurance companies need to estimate claim payouts to set appropriate premiums and manage risk. By predicting claim values based on policyholder and vehicle characteristics, insurers can:
- Price policies more accurately
- Identify high-risk factors
- Allocate resources more efficiently

### What We'll Do
1. **Load and validate data** - Understand the raw dataset
2. **Clean and prepare data** - Handle missing values and remove problematic features
3. **Engineer features** - Select the best variables for prediction
4. **Build preprocessing pipelines** - Standardize numeric features and encode categorical ones
5. **Train 7 different models** - Compare multiple machine learning approaches
6. **Track experiments with MLflow** - Keep detailed records of each model's performance
7. **Save and deploy the best model** - Export it for production use

### Dataset: beMTPL16
This Belgian motor third-party liability dataset contains **70,791 insurance claims** with 19 variables describing the policyholder, vehicle, and claim circumstances.

## Quick Start Guide

**First time running this notebook?** Here's the workflow:

1. **Run all cells in order** - They depend on previous steps
2. **Don't worry if you see warnings** - MLflow warnings about signatures are expected and harmless
3. **When training completes**, section 5 shows which model performed best
4. **View results in MLflow dashboard**: `mlflow ui` in terminal, then http://localhost:5000
5. **Optional: Deploy to cloud** - Section 7 uploads your model to Hugging Face Hub

### What Makes This Notebook Educational

- **Every section has markdown explanations** - Not just code comments
- **Real-world problem** - Insurance claim prediction (not toy datasets)
- **Best practices included** - Experiment tracking, preprocessing pipelines, model comparison
- **Production-ready** - Can be deployed to Streamlit or any web framework

### Common Questions

**Q: Do I need a GPU?**
A: No, this runs fine on CPU. Models may take 5-10 minutes to train.

**Q: What if a model fails to train?**
A: Each model trains independently. If one fails, the others still work. Check the error message.

**Q: Can I modify the features?**
A: Yes! Change the `features` list in Section 2 to experiment with different variables.

**Q: How do I make predictions on new data?**
A: Load the best model from `./best_model_backup/` and follow the example in Section 6.

---



## Data Loading

The dataset is stored as an **R data file (.rda)**, which is a compressed format commonly used in the R programming language. We need special tools to read it in Python.

**What happens in the code below:**
1. Install `pyreadr` - a library that can read R files
2. Download the beMTPL16 dataset from the CAS (Casualty Actuarial Society) datasets repository
3. Cache the file locally so we don't re-download it on every run
4. Convert it to a Pandas DataFrame so we can work with it in Python
5. Display the first few rows to verify it loaded correctly

In [ ]:
%pip install pyreadr -q
import requests
import os
import pyreadr
import pandas as pd
import numpy as np

raw_url = 'https://github.com/dutangc/CASdatasets/raw/master/data/beMTPL16.rda'
file_name = os.path.basename(raw_url)
print(f"Downloading {file_name}...")

try:
    if not os.path.exists(file_name):
        response = requests.get(raw_url)
        response.raise_for_status()
        with open(file_name, 'wb') as f:
            f.write(response.content)
        print("Downloaded successfully")
    else:
        print("Using cached file")

    result = pyreadr.read_r(file_name)
    df_beMTPL16 = result['beMTPL16']
    print(f"Data loaded: {df_beMTPL16.shape[0]} rows, {df_beMTPL16.shape[1]} columns")
except Exception as e:
    print(f"Error: {e}")


Using cached file
Using cached file
Data loaded: 70791 rows, 19 columns
Data loaded: 70791 rows, 19 columns


In [2]:
display(df_beMTPL16.head())

,insurance_contract,policy_year,exposure,insured_birth_year,vehicle_age,policy_holder_age,driver_license_age,vehicle_brand,vehicle_model,mileage,vehicle_power,catalog_value,claim_value,number_of_liability_claims,number_of_bodily_injury_liability_claims,claim_time,claim_responsibility_rate,driving_training_label,signal
0,C1,1,0.386301,1945,10,9,40,MERCEDES,ME-1245,30000,75,983732,2,0,0,00:00,0,No,0
1,C2,1,0.493151,1941,4,25,24,VOLKSWAGEN,VO-2461,30000,55,510562,8,0,0,07:45,0,No,0
2,C3,1,0.290411,1944,0,2,39,AUDI,AU-967,30000,120,1934768,10,0,0,00:00,0,No,0
3,C4,1,0.336986,1948,1,14,37,LANCIA,LA-2346,30000,51,536755,13,0,0,18:50,0,No,0
4,C5,1,0.219178,1928,3,7,59,CITROEN,CI-1258,30000,54,446725,14,0,0,00:00,100,No,0


## 1. Data Validation & Understanding

### What Is Data Validation?
Before we build a machine learning model, we need to understand our data thoroughly:
- **Data types** - Are columns numeric, categorical, or text?
- **Missing values** - How many cells are empty or zero?
- **Distributions** - What are the min, max, and average values?
- **Quality issues** - Are there suspicious patterns or outliers?

This process is called *exploratory data analysis (EDA)* and is critical for building good models.

### Complete Dataset Schema

| Column Name                                 | Type     | Description                                                         |

### Data Inspection Tools

We use two key Pandas methods to understand our data:

**`info()` Method**
- Shows the number of rows and columns
- Lists the data type of each column
- Counts non-null values (helps identify missing data)
- Reports memory usage

Example: If a column shows "1000 non-null" out of "1020 total", that means 20 values are missing.

**`describe()` Method**
- Shows statistical summaries: mean, median (50%), min, max, quartiles
- Only works on numeric columns
- Helps spot outliers or unusual distributions

Example: If average claim value is $5,000 but the max is $500,000, there are some very expensive claims.

**Why This Matters**: Missing data and outliers can hurt model accuracy, so we need to handle them before training.

In [3]:
print("Dataset Overview")
df_beMTPL16.info()

print("\nNull Values:")
print(df_beMTPL16.isnull().sum())

cat_col = [col for col in df_beMTPL16.columns if df_beMTPL16[col].dtype == 'category']
num_col = [col for col in df_beMTPL16.columns if df_beMTPL16[col].dtype != 'category']

print(f'\nColumns: {len(cat_col)} categorical, {len(num_col)} numerical')

zero_counts = {col: (df_beMTPL16[col] == 0).sum() for col in num_col}
zero_df = pd.DataFrame([(col, count, f"{count/len(df_beMTPL16)*100:.2f}%") 
                         for col, count in zero_counts.items() if count > 0],
                       columns=['Column', 'Zero Count', 'Percentage'])
print("\nZero Values:")
display(zero_df.sort_values(by='Zero Count', ascending=False))

print("\nStatistics:")
display(df_beMTPL16.describe())


Dataset Overview
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70791 entries, 0 to 70790
Data columns (total 19 columns):
 #   Column                                    Non-Null Count  Dtype   
---  ------                                    --------------  -----   
 0   insurance_contract                        70791 non-null  category
 1   policy_year                               70791 non-null  int32   
 2   exposure                                  70791 non-null  float64 
 3   insured_birth_year                        70791 non-null  int32   
 4   vehicle_age                               70791 non-null  int32   
 5   policy_holder_age                         70791 non-null  int32   
 6   driver_license_age                        70791 non-null  int32   
 7   vehicle_brand                             70791 non-null  category
 8   vehicle_model                             70791 non-null  category
 9   mileage                                   70791 non-null  int32   
 10  vehic

,Column,Zero Count,Percentage
7,signal,70746,99.94%
5,number_of_bodily_injury_liability_claims,69381,98.01%
4,number_of_liability_claims,46080,65.09%
6,claim_responsibility_rate,36017,50.88%
3,catalog_value,21844,30.86%
0,vehicle_age,3304,4.67%
1,policy_holder_age,2331,3.29%
2,driver_license_age,3,0.00%



Statistics:


,policy_year,exposure,insured_birth_year,vehicle_age,policy_holder_age,driver_license_age,mileage,vehicle_power,catalog_value,claim_value,number_of_liability_claims,number_of_bodily_injury_liability_claims,claim_responsibility_rate,signal
count,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,7.079100e+04,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000
mean,2.503934,0.437601,1941.394231,6.163849,9.651679,38.196875,28327.824158,77.962382,5.823853e+05,80780.786371,0.349070,0.019918,48.424362,0.000636
std,1.108830,0.180683,6.960452,4.803108,6.789322,9.549636,5711.127945,29.633158,5.460070e+05,45999.887220,0.476679,0.139719,49.628070,0.025205
min,1.000000,0.200000,1911.000000,0.000000,0.000000,0.000000,2500.000000,30.000000,0.000000e+00,2.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,0.287671,1936.000000,2.000000,4.000000,34.000000,30000.000000,55.000000,0.000000e+00,41491.500000,0.000000,0.000000,0.000000,0.000000
50%,3.000000,0.400000,1943.000000,5.000000,9.000000,41.000000,30000.000000,74.000000,5.506000e+05,82430.000000,0.000000,0.000000,0.000000,0.000000
75%,3.000000,0.556164,1947.000000,9.000000,14.000000,43.000000,30000.000000,92.000000,8.677665e+05,120472.000000,1.000000,0.000000,100.000000,0.000000
max,4.000000,1.000000,1952.000000,60.000000,30.000000,60.000000,30000.000000,487.000000,7.234528e+06,169694.000000,1.000000,1.000000,100.000000,1.000000


## Data Cleaning Strategy

### Why Remove Columns?
Not all columns are useful for prediction. Some are:
- **Redundant**: They say the same thing as another column (e.g., age and birth year)
- **Identifiers**: They don't predict anything, just uniquely identify records
- **Data leakage**: They contain information that's only available *after* the claim is processed (unfair to use for prediction)
- **No variance**: Nearly all values are the same (no predictive power)

Removing these columns makes the model simpler, faster, and more reliable.

### Columns to Remove

> **Note on Column Names:** The GitHub documentation refers to `insured_year_birth`, but the actual R dataset uses `insured_birth_year`. We use the real data column name (`insured_birth_year`) in this notebook.

| Column Name | Reason to Drop |

In [ ]:
print("Data Cleaning")

columns_to_drop = [
    'insured_birth_year',
    'driving_training_label',
    'record_number'
]

df_model_data = df_beMTPL16.drop(columns=columns_to_drop, errors='ignore')
df_model_data = df_model_data[df_model_data['claim_value'] > 0].copy()

print(f"Dataset after cleaning: {df_model_data.shape[0]} rows (removed {len(df_beMTPL16) - df_model_data.shape[0]} rows with zero/negative claims)")

numeric_cols_with_zeros = ['policy_holder_age', 'driver_license_age', 'catalog_value']

for col in numeric_cols_with_zeros:
    if col in df_model_data.columns:
        zero_count = (df_model_data[col] == 0).sum()
        if zero_count > 0:
            median_value = df_model_data[df_model_data[col] > 0][col].median()
            df_model_data.loc[df_model_data[col] == 0, col] = median_value
            print(f"  {col}: Replaced {zero_count} zeros with median {median_value:.2f}")

print(f"\nFinal dataset: {df_model_data.shape[0]} rows, {df_model_data.shape[1]} columns")


| `driving_training_label` | **Low Real-World Value:** Sparse indicator with limited predictive power; excluded for model simplicity. |

---

### Columns to Keep

After removing problematic columns, we're left with features that are **predictive, non-redundant, and not subject to data leakage**. These will form our machine learning model.

#### Our Prediction Target

This is the **one number** our model will learn to predict.

| Column Name | Role | Notes |
| :--- | :--- | :--- |
| `claim_value` | **Target (Label)** | The claim amount in EUR that our model will try to predict. We only use claims where `claim_value > 0` (real claims, not zero-value entries). |

#### Input Features

These are the **10 variables** the model will use as clues to predict claim value.

| Column Name | Role | Notes |
| :--- | :--- | :--- |
| `policy_year` | Feature | Year of the policy - helps capture inflation trends over time. |
| `vehicle_age` | Feature | Older vehicles may have different claim patterns. Zeros (4.67%) likely mean "new car". |
| `policy_holder_age` | Feature | Younger drivers often have more claims. **Issue:** 3.29% zeros = missing data that needs filling. |
| `driver_license_age` | Feature | Inexperienced drivers may have different claim behavior. **Issue:** A few zeros need to be fixed. |
| `vehicle_brand` | Feature | Some brands may be safer or more expensive to repair. **Categorical** - needs encoding. |
| `vehicle_model` | Feature | Like brand, model affects cost and claims. **Categorical** - needs encoding. |
| `mileage` | Feature | Higher mileage vehicles may be older or driven more aggressively. |
| `vehicle_power` | Feature | More powerful vehicles may correlate with performance driving and higher claims. |
| `catalog_value` | Feature | Expensive cars cost more to repair. **Issue:** 30.86% zeros = missing data, needs smart filling. |
| `claim_time` | Feature | Accidents at night may differ from day. **Categorical** - needs encoding. |

## 2. Feature Engineering & Data Preparation

### What Is Feature Engineering?
Feature engineering is the process of:
1. **Selecting which variables matter** - We chose 10 features that are predictive and not redundant
2. **Creating the training set** - Split data into training (80%) and testing (20%) portions
3. **Transform the target** - Apply mathematical transformations to make the model's job easier

### Why Log-Transform the Target?
Claim values are **right-skewed**: most claims are small, but a few are huge. This shape confuses machine learning models.

Using `log(claim_value)` instead:
- Flattens the distribution to bell-curve shape
- Makes relationships more linear
- Reduces impact of outliers
- Improves model accuracy

**Example**: If one claim is €100 and another is €10,000, the ratio is 100x. In log space, it's only ~4.6x difference - more balanced.

After predictions, we convert back using `exp()` to get actual EUR values that make sense.

### Train-Test Split
We always split data into:
- **Training set (80%)**: Used to teach the model
- **Test set (20%)**: Used to evaluate how well it works on unseen data

This prevents the model from "cheating" by memorizing the test data.



In [ ]:
from sklearn.model_selection import train_test_split

print("Feature Engineering")

features = [
    'policy_year', 'vehicle_age', 'policy_holder_age', 'driver_license_age',
    'vehicle_brand', 'vehicle_model', 'mileage', 'vehicle_power',
    'catalog_value', 'claim_time'
]
target = 'claim_value'

X = df_model_data[features]
y = np.log1p(df_model_data[target])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")


Feature Engineering
Train: 56630 rows, Test: 14158 rows


## 3. Preprocessing: Preparing Data for Machine Learning

### Why Preprocessing Matters
Machine learning models work best when data is in the right format:
- **Numeric features** need to be on the same scale (a person's age shouldn't dominate mileage just because the numbers are bigger)
- **Categorical features** (brand, model) need to be converted to numbers

### Our Two-Part Preprocessing Strategy

#### Part 1: StandardScaler (Numeric Features)
**Problem**: Variables have different ranges
- `policy_year`: 2004-2008 (range ~4)
- `mileage`: 0-100,000+ (range very large)
- `vehicle_power`: 0-500+ (range large)

**Solution**: StandardScaler converts all numbers to a standard scale (mean=0, range ±3) so no variable dominates.

**Example**: Instead of mileage values like 5,000, we transform to z-score like -0.5 or +1.2

#### Part 2: OneHotEncoder (Categorical Features)
**Problem**: Models can't work with text like "Toyota" or "BMW"

**Solution**: OneHotEncoder converts each category into binary (0/1) columns

**Example**: Instead of one "vehicle_brand" column with text values:
```
Before:           After:
vehicle_brand     vehicle_brand_BMW  vehicle_brand_Toyota  vehicle_brand_Audi
BMW       →       1                  0                     0
Toyota    →       0                  1                     0  
Audi      →       0                  0                     1
```

### Two Different Pipelines
We build **two different preprocessing pipelines**:
1. **OneHot Pipeline** (for Models 1-4): Uses OneHotEncoder for all categories
2. **Target Encoding Pipeline** (for Actuarial Model 5): Uses Target Encoding for high-cardinality features

Target Encoding works better for high-cardinality categorical variables (like vehicle brands/models with many options).



In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

print("Preprocessing Pipeline")

numeric_features = [
    'policy_year', 'vehicle_age', 'policy_holder_age',
    'driver_license_age', 'mileage', 'vehicle_power', 'catalog_value'
]
categorical_features = ['vehicle_brand', 'vehicle_model', 'claim_time']

numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01, sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

processed_feature_names = preprocessor.get_feature_names_out()
print(f"Processed features: {X_train_processed.shape[1]}")

print("\n\nActuarial XGBoost Preprocessing Pipeline (with Target Encoding)")
%pip install category_encoders -q
import category_encoders
from category_encoders import TargetEncoder

y_train_original = np.expm1(y_train)

numeric_transformer_actuarial = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer_actuarial = ColumnTransformer(
    transformers=[
        ('target_high', TargetEncoder(), ['vehicle_brand', 'vehicle_model']),
        ('onehot_low', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['claim_time'])
    ],
    remainder='drop'
)

preprocessor_actuarial = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_actuarial, numeric_features),
        ('cat', categorical_transformer_actuarial, categorical_features)
    ],
    remainder='drop'
)

preprocessor_actuarial.fit(X_train, y_train_original)
X_train_actuarial = preprocessor_actuarial.transform(X_train)
X_test_actuarial = preprocessor_actuarial.transform(X_test)

print(f"Actuarial processed features: {X_train_actuarial.shape[1]}")
print("Target encoding applied to vehicle_brand and vehicle_model")


Preprocessing Pipeline
Processed features: 67


Actuarial XGBoost Preprocessing Pipeline (with Target Encoding)
category_encoders is already installed
category_encoders is already installed
Actuarial processed features: 1005
Target encoding applied to vehicle_brand and vehicle_model
Actuarial processed features: 1005
Target encoding applied to vehicle_brand and vehicle_model


## 4. Model Training & Comparison

### Why Train Multiple Models?
Different machine learning algorithms have different strengths:
- Some are simpler but may underfit
- Some are complex but may overfit
- Some work better with certain data patterns

By training 7 different models, we can compare and pick the best one.

### The Seven Models We'll Train

**Model 1: Linear Regression (Baseline)**
- **How it works**: Finds a straight line (or plane in higher dimensions) that best fits the data
- **Pros**: Simple, interpretable, fast
- **Cons**: Can't capture complex non-linear patterns
- **Use case**: Baseline to compare other models against

**Model 2: XGBoost (Basic)**
- **How it works**: Builds a series of decision trees sequentially, each one correcting the previous one's mistakes
- **Pros**: Usually performs very well, handles non-linear patterns
- **Cons**: More complex, harder to interpret
- **Use case**: Standard gradient boosting approach

**Model 3: Random Forest**
- **How it works**: Trains many decision trees in parallel on random data subsets, then averages their predictions
- **Pros**: Robust, handles outliers well, captures non-linear patterns
- **Cons**: Can be slow, harder to interpret
- **Use case**: When robustness and stability matter

**Model 4: XGBoost (Tuned)**
- **How it works**: XGBoost with optimized hyperparameters (found via RandomizedSearchCV)
- **Pros**: Should perform better than basic XGBoost
- **Cons**: Takes longer to train due to hyperparameter search
- **Use case**: When we want the best possible performance

**Model 5: Actuarial XGBoost (Gamma Regression)**
- **How it works**: XGBoost trained on original (non-log) claim values using Gamma regression
- **Pros**: Specifically designed for insurance claim severity, respects the right-skewed distribution
- **Cons**: Less standard, fewer people use it
- **Use case**: When domain knowledge (actuarial science) is important

### Performance Metrics
We'll evaluate all models using:
- **RMSE (Root Mean Squared Error)** in EUR - How far off are predictions on average? Lower is better.
- **MAPE (Mean Absolute Percentage Error)** - Percentage error. If actual is €1000 and we predict €900, that's 10% error.

### 4.1 Linear Regression Baseline

We start with Linear Regression as our **baseline model**—a simple reference point to judge whether the more complex models are actually worth the added complexity.


## Experiment Tracking with MLflow

### What Is MLflow and Why Use It?
When training machine learning models, you'll often run the same code many times with different parameters. Without proper tracking, it's easy to lose track of:
- What hyperparameters did you use?
- What was the model's performance?
- Which model was the best?
- Can you reproduce your results?

**MLflow** is an open-source tool that automatically tracks:
- **Parameters**: All settings you changed (learning rate, tree depth, etc.)
- **Metrics**: Performance numbers (RMSE, MAPE, accuracy)
- **Model Artifacts**: The actual trained model files
- **Runs**: Each experiment execution
- **Experiments**: Collections of related runs

### How We Use MLflow
Each time we train a model, we:
1. Start a new MLflow run with a descriptive name
2. Automatically log hyperparameters
3. Manually log business metrics (RMSE in EUR, MAPE)
4. Save the trained model as an artifact
5. View everything in the MLflow dashboard (http://localhost:5000)

### Our Configuration
- **Tracking URI**: `./mlruns` (local folder on your computer)
- **Experiment**: "Belgian_MTPL_Severity"
- **Autologging**: Enabled for sklearn and XGBoost models

This means you'll have a complete record of every model training run, making it easy to:
- Compare models
- Reproduce results
- Deploy the best model
- Share results with team members



In [ ]:
%pip install mlflow -q

import mlflow
import mlflow.sklearn
import mlflow.xgboost

# Set tracking URI (local file system)
mlflow.set_tracking_uri("./mlruns")

experiment_name = "Belgian_MTPL_Severity"
try:
    experiment_id = mlflow.create_experiment(experiment_name)
except mlflow.exceptions.MlflowException:
    # Experiment already exists
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

mlflow.set_experiment(experiment_name)

mlflow.sklearn.autolog(log_input_examples=False, log_model_signatures=False)
mlflow.xgboost.autolog(log_input_examples=False, log_model_signatures=False)

print(f"MLflow configured successfully!")
print(f"Experiment: {experiment_name}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Run logs will be saved to ./mlruns directory")


/Users/charlesnanakwakye/HobbyApps/be-insurance-ai/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)


MLflow configured successfully!
Experiment: Belgian_MTPL_Severity
Tracking URI: ./mlruns
Run logs will be saved to ./mlruns directory


### Optional: Clean Up Old MLflow Runs

Each time you run this notebook, a new MLflow run is created. If you run it many times, the `./mlruns` folder will grow large with old experiments.

**When to clean up**:
- You want to start fresh with an empty experiment history
- The folder is taking up disk space
- You want to remove outdated experiments

**How to use this cell**:
1. Set `CLEAN_MLFLOW_RUNS = True` to delete all old runs
2. Run the cell
3. Immediately change it back to `False` so you don't accidentally delete runs next time
4. Continue with training models

This is optional - you can also keep all runs to see your progress over time.



In [ ]:
import shutil
import os

CLEAN_MLFLOW_RUNS = True

if CLEAN_MLFLOW_RUNS:
    mlruns_path = "./mlruns"
    if os.path.exists(mlruns_path):
        shutil.rmtree(mlruns_path)
        print("Cleaned up old MLflow runs directory")
        print("All previous experiment data has been deleted.")
    else:
        print("No MLflow runs directory found")
else:
    print("Skipping cleanup. To clean up old runs, set CLEAN_MLFLOW_RUNS = True above")


In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="Baseline_Linear_Regression"):
    model_lr = LinearRegression()
    model_lr.fit(X_train_processed, y_train)
    pred_lr = model_lr.predict(X_test_processed)
    test_rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    pred_lr_original = np.expm1(pred_lr)
    rmse_lr_eur = np.sqrt(mean_squared_error(y_test_original, pred_lr_original))
    mape_lr = mean_absolute_percentage_error(y_test_original, pred_lr_original)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_lr_eur)
    mlflow.log_metric("mape", mape_lr)
    
    print(f"Linear Regression RMSE (log scale): {test_rmse_lr:.6f}")
    print(f"  RMSE (EUR): €{rmse_lr_eur:.2f}")
    print(f"  MAPE: {mape_lr * 100:.2f}%")

2025/11/22 02:41:51 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/11/22 02:41:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:41:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Linear Regression RMSE (log scale): 0.504837
  RMSE (EUR): €24445.42
  MAPE: 59.66%


### 4.2 XGBoost (Basic Configuration)

Now we train XGBoost, a powerful gradient boosting algorithm. This model builds many decision trees sequentially, with each tree learning to correct the previous trees' mistakes.

**Key settings**:
- **n_estimators=1000**: Maximum number of trees to build (we'll probably use fewer due to early stopping)
- **early_stopping_rounds=50**: Stop training if validation performance doesn't improve for 50 rounds

**Why early stopping matters**: It prevents the model from overfitting (memorizing the training data instead of learning general patterns).


In [9]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="XGBoost_Basic"):
    model_xgb = xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=5,
        early_stopping_rounds=50,
        random_state=42
    )
    model_xgb.fit(
        X_train_processed, y_train,
        eval_set=[(X_test_processed, y_test)],
        verbose=False
    )
    pred_xgb = model_xgb.predict(X_test_processed)
    test_rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    pred_xgb_original = np.expm1(pred_xgb)
    rmse_xgb_eur = np.sqrt(mean_squared_error(y_test_original, pred_xgb_original))
    mape_xgb = mean_absolute_percentage_error(y_test_original, pred_xgb_original)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_xgb_eur)
    mlflow.log_metric("mape", mape_xgb)
    
    print(f"XGBoost (Basic) RMSE (log scale): {test_rmse_xgb:.6f}")
    print(f"  RMSE (EUR): €{rmse_xgb_eur:.2f}")
    print(f"  MAPE: {mape_xgb * 100:.2f}%")

2025/11/22 02:41:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/22 02:41:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:41:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


XGBoost (Basic) RMSE (log scale): 0.428026
  RMSE (EUR): €12301.53
  MAPE: 40.94%


### 4.3 Random Forest

Random Forest is another ensemble method: it trains 200 independent decision trees on random subsets of the data, then averages their predictions.

**Why this approach is robust**:
- Each tree is trained on slightly different data
- Trees make different mistakes
- Averaging reduces individual errors
- Result: A model that works well even on new data

We use `n_jobs=-1` to train all trees in parallel (faster on multi-core computers).


In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="RF_Ensemble"):
    model_rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    model_rf.fit(X_train_processed, y_train)
    pred_rf = model_rf.predict(X_test_processed)
    test_rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    pred_rf_original = np.expm1(pred_rf)
    rmse_rf_eur = np.sqrt(mean_squared_error(y_test_original, pred_rf_original))
    mape_rf = mean_absolute_percentage_error(y_test_original, pred_rf_original)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_rf_eur)
    mlflow.log_metric("mape", mape_rf)
    
    print(f"Random Forest RMSE (log scale): {test_rmse_rf:.6f}")
    print(f"  RMSE (EUR): €{rmse_rf_eur:.2f}")
    print(f"  MAPE: {mape_rf * 100:.2f}%")

2025/11/22 02:41:57 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/11/22 02:42:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:42:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Random Forest RMSE (log scale): 0.427920
  RMSE (EUR): €12309.59
  MAPE: 40.20%


### 4.4 XGBoost with Hyperparameter Tuning

While Basic XGBoost (Model 2) uses default settings, this model **searches for optimal settings**.

**What are hyperparameters?**
These are settings you choose *before* training (unlike model weights which are learned during training):
- `learning_rate`: How much each tree contributes (smaller = slower but more stable)
- `max_depth`: How deep each tree can grow (deeper = more complex)
- `subsample`: Percentage of data each tree sees (less = more robust but slower)

**How RandomizedSearchCV works**:
1. Randomly try 50 different combinations of hyperparameters
2. For each combination, use 5-fold cross-validation to evaluate it
3. Keep the combination that performs best

This is computationally expensive but often finds much better settings than defaults.


In [11]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="XGB_Tuned_RandomSearch"):
    param_dist = {
        'n_estimators': randint(200, 1500),
        'learning_rate': uniform(0.01, 0.1),
        'max_depth': randint(3, 8),
        'subsample': uniform(0.7, 0.3),
        'colsample_bytree': uniform(0.7, 0.3)
    }

    random_search = RandomizedSearchCV(
        xgb.XGBRegressor(random_state=42),
        param_distributions=param_dist,
        n_iter=50,
        cv=5,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    random_search.fit(X_train_processed, y_train)
    best_model_xgb = random_search.best_estimator_
    log_predictions_tuned = best_model_xgb.predict(X_test_processed)
    final_test_rmse = np.sqrt(mean_squared_error(y_test, log_predictions_tuned))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    predictions_original = np.expm1(log_predictions_tuned)
    rmse_tuned_eur = np.sqrt(mean_squared_error(y_test_original, predictions_original))
    mape_tuned = mean_absolute_percentage_error(y_test_original, predictions_original)
    
    # Log best hyperparameters
    mlflow.log_params(random_search.best_params_)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_tuned_eur)
    mlflow.log_metric("mape", mape_tuned)

    print(f"XGBoost (Tuned) RMSE (log scale): {final_test_rmse:.6f}")
    print(f"  RMSE (EUR): €{rmse_tuned_eur:.2f}")
    print(f"  MAPE: {mape_tuned * 100:.2f}%")
    print(f"\nBest Hyperparameters:")
    for param, value in random_search.best_params_.items():
        print(f"  {param}: {value}")

2025/11/22 02:42:07 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/11/22 02:47:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:47:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:47:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:47:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:47:13 INFO mlflow.sklearn.utils: Logging the

XGBoost (Tuned) RMSE (log scale): 0.427424
  RMSE (EUR): €12336.02
  MAPE: 40.62%

Best Hyperparameters:
  colsample_bytree: 0.8023199053150775
  learning_rate: 0.02134735212405891
  max_depth: 4
  n_estimators: 462
  subsample: 0.8979952138102536


### 4.5 Actuarial XGBoost (Gamma Regression)

This is the most specialized model, designed by actuaries specifically for insurance claim severity.

**Why a different approach?**
Most models assume errors are normally distributed (bell curve). Claim amounts follow a **Gamma distribution** (right-skewed: many small claims, few huge ones).

**Gamma regression key differences**:
- `objective='reg:gamma'`: Uses Gamma loss function designed for right-skewed data
- **Original scale targets**: Uses claim values as-is (no log transformation)
- **Why this matters**: The model directly learns the right-skewed pattern instead of trying to transform it away

**Why Target Encoding matters here**:
Vehicle brands and models are "high-cardinality" (lots of unique values). Target Encoding encodes each category by its average claim value:
- BMW brands average €3,500 claims → replace "BMW" with 3500
- Audi brands average €3,200 claims → replace "Audi" with 3200

This preserves the relationship between vehicle type and claim severity.

**Expected advantage**: May perform better than log-scale models because it respects the true distribution of insurance claims.


### 4.6 Ridge Regression

Ridge Regression is a variation of Linear Regression that adds a **penalty for large coefficients**. This prevents the model from overfitting by discouraging it from relying too heavily on any single feature.

**How it works**:
- Like Linear Regression, it finds weights for each feature
- But it penalizes models where weights are too large
- The `alpha` parameter controls how strong the penalty is

**Why this is useful**:
- **More stable**: Less sensitive to extreme feature values
- **Better generalization**: Often performs better on test data than regular Linear Regression
- **Handles multicollinearity**: When features are correlated with each other

**When to use Ridge**:
- You suspect the Linear model is overfitting
- Features are correlated with each other
- You want something between pure Linear Regression and complex models

**Regularization penalty**: L2 (sum of squared weights) - gentle penalty that shrinks all weights proportionally


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="Ridge_Regression"):
    model_ridge = Ridge(alpha=1.0, random_state=42)
    model_ridge.fit(X_train_processed, y_train)
    pred_ridge = model_ridge.predict(X_test_processed)
    test_rmse_ridge = np.sqrt(mean_squared_error(y_test, pred_ridge))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    pred_ridge_original = np.expm1(pred_ridge)
    rmse_ridge_eur = np.sqrt(mean_squared_error(y_test_original, pred_ridge_original))
    mape_ridge = mean_absolute_percentage_error(y_test_original, pred_ridge_original)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_ridge_eur)
    mlflow.log_metric("mape", mape_ridge)
    
    print(f"Ridge Regression RMSE (log scale): {test_rmse_ridge:.6f}")
    print(f"  RMSE (EUR): €{rmse_ridge_eur:.2f}")
    print(f"  MAPE: {mape_ridge * 100:.2f}%")


### 4.7 Lasso Regression

Lasso Regression is another variation of Linear Regression that also adds a **penalty for large coefficients**, but in a different way than Ridge.

**How it works**:
- Like Ridge, it penalizes large weights
- But Lasso can force some weights to **exactly zero**
- This means Lasso automatically performs **feature selection**
- Useful for identifying the most important features

**Key difference from Ridge**:
- **Ridge**: Shrinks all weights, but keeps them all
- **Lasso**: Can eliminate less important features completely

**Why this is useful**:
- **Interpretability**: Identifies which features matter most
- **Feature selection**: Automatically removes irrelevant features
- **Simplicity**: Models with fewer features are easier to understand

**When to use Lasso**:
- You have many features and want to know which ones matter
- You suspect some features are completely irrelevant
- You want a simpler, more interpretable model

**Regularization penalty**: L1 (sum of absolute weights) - strong penalty that can force weights to zero

**Example**: If you have 50 features, Lasso might reduce it to 15 important ones, making the model simpler.


In [ ]:
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="Lasso_Regression"):
    model_lasso = Lasso(alpha=0.1, random_state=42, max_iter=5000)
    model_lasso.fit(X_train_processed, y_train)
    pred_lasso = model_lasso.predict(X_test_processed)
    test_rmse_lasso = np.sqrt(mean_squared_error(y_test, pred_lasso))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    pred_lasso_original = np.expm1(pred_lasso)
    rmse_lasso_eur = np.sqrt(mean_squared_error(y_test_original, pred_lasso_original))
    mape_lasso = mean_absolute_percentage_error(y_test_original, pred_lasso_original)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_lasso_eur)
    mlflow.log_metric("mape", mape_lasso)
    
    # Show feature selection (how many features were zeroed out)
    non_zero_features = (model_lasso.coef_ != 0).sum()
    total_features = len(model_lasso.coef_)
    
    print(f"Lasso Regression RMSE (log scale): {test_rmse_lasso:.6f}")
    print(f"  RMSE (EUR): €{rmse_lasso_eur:.2f}")
    print(f"  MAPE: {mape_lasso * 100:.2f}%")
    print(f"  Feature Selection: {non_zero_features}/{total_features} features kept (removed {total_features - non_zero_features})")


In [12]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="Actuarial_XGB_Gamma"):
    # Critical: Use original claim values (not log-transformed) for Gamma regression
    y_train_original = np.expm1(y_train)
    y_test_original = np.expm1(y_test)

    # Train Actuarial XGBoost with Gamma regression
    model_xgb_actuarial = xgb.XGBRegressor(
        objective='reg:gamma',
        n_estimators=500,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1
    )

    model_xgb_actuarial.fit(
        X_train_actuarial, y_train_original,
        eval_set=[(X_test_actuarial, y_test_original)],
        verbose=False
    )

    # Predict on test set
    pred_xgb_actuarial = model_xgb_actuarial.predict(X_test_actuarial)

    # Evaluate on original scale
    rmse_actuarial = np.sqrt(mean_squared_error(y_test_original, pred_xgb_actuarial))
    mape_actuarial = mean_absolute_percentage_error(y_test_original, pred_xgb_actuarial)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_actuarial)
    mlflow.log_metric("mape", mape_actuarial)

    print(f"Actuarial XGBoost (Gamma Regression) Performance:")
    print(f"  RMSE (original scale, EUR): €{rmse_actuarial:.2f}")
    print(f"  MAPE: {mape_actuarial * 100:.2f}%")

2025/11/22 02:47:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/22 02:47:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:47:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Actuarial XGBoost (Gamma Regression) Performance:
  RMSE (original scale, EUR): €12113.84
  MAPE: 50.69%


## 5. Results Analysis & Model Comparison

### What We're About to Do
Now we'll:
1. **Compare all 7 models** side-by-side using the same metrics
2. **Identify the best model** based on test set performance
3. **Show predictions vs actual** for the first 20 test claims
4. **Analyze errors** to understand where the models struggle
5. **Generate statistics** on prediction accuracy

### Why This Matters
Before deploying a model to production, we need to:
- Ensure it actually works better than simple baselines
- Understand its limitations and weaknesses
- Know what error rates to expect
- Be confident it generalizes to new data

### Metrics Explained

**RMSE (Root Mean Squared Error)** - in EUR
- Average amount our predictions are off by
- Example: RMSE of €500 means predictions are typically off by ±€500
- Lower is better
- Penalizes large errors more than small ones

**MAPE (Mean Absolute Percentage Error)** - in percent
- Average error as a percentage of actual claim value
- Example: If actual claim is €1,000 and we predict €900, error is 10%
- Lower is better
- Better for comparing errors across different claim amounts



In [ ]:
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd

y_test_original = np.expm1(y_test)

model_configs = {
    'Linear Regression': {
        'rmse': lambda: np.sqrt(mean_squared_error(y_test_original, np.expm1(pred_lr))),
        'mape': lambda: mean_absolute_percentage_error(y_test_original, np.expm1(pred_lr)),
        'pred': lambda: np.expm1(pred_lr)
    },
    'XGBoost (basic)': {
        'rmse': lambda: np.sqrt(mean_squared_error(y_test_original, np.expm1(pred_xgb))),
        'mape': lambda: mean_absolute_percentage_error(y_test_original, np.expm1(pred_xgb)),
        'pred': lambda: np.expm1(pred_xgb)
    },
    'Random Forest': {
        'rmse': lambda: np.sqrt(mean_squared_error(y_test_original, np.expm1(pred_rf))),
        'mape': lambda: mean_absolute_percentage_error(y_test_original, np.expm1(pred_rf)),
        'pred': lambda: np.expm1(pred_rf)
    },
    'XGBoost (tuned)': {
        'rmse': lambda: rmse_tuned_eur,
        'mape': lambda: mape_tuned,
        'pred': lambda: np.expm1(log_predictions_tuned)
    },
    'Actuarial XGBoost': {
        'rmse': lambda: rmse_actuarial,
        'mape': lambda: mape_actuarial,
        'pred': lambda: pred_xgb_actuarial
    },
    'Ridge Regression': {
        'rmse': lambda: rmse_ridge_eur,
        'mape': lambda: mape_ridge,
        'pred': lambda: pred_ridge_original
    },
    'Lasso Regression': {
        'rmse': lambda: rmse_lasso_eur,
        'mape': lambda: mape_lasso,
        'pred': lambda: pred_lasso_original
    }
}

models_rmse = {}
models_mape = {}
models_pred = {}

for name, config in model_configs.items():
    try:
        models_rmse[name] = config['rmse']()
        models_mape[name] = config['mape']()
        models_pred[name] = config['pred']()
    except NameError:
        pass

model_list = list(models_rmse.keys())

print("=" * 80)
print(f"MODEL COMPARISON: {len(model_list)} MODELS")
print("=" * 80)
print("\nNote: Detailed metrics, hyperparameters, and model artifacts are now")
print("tracked in MLflow. View the dashboard with: mlflow ui")
print("=" * 80)

print("\nModel Performance (RMSE on original scale, EUR):")
for i, name in enumerate(model_list, 1):
    print(f"  {i}. {name:<30} €{models_rmse[name]:.2f}")

print("\nModel Performance (MAPE):")
for i, name in enumerate(model_list, 1):
    print(f"  {i}. {name:<30} {models_mape[name] * 100:.2f}%")

print("\n" + "=" * 80)
print("BEST MODEL SUMMARY")
print("=" * 80)

best_model = min(models_rmse, key=models_rmse.get)
print(f"\nBest Model: {best_model}")
print(f"  RMSE: €{models_rmse[best_model]:.2f}")
print(f"  MAPE: {models_mape[best_model] * 100:.2f}%")
print(f"  Average Claim Value: €{y_test_original.mean():.2f}")

print("\n" + "=" * 80)
print("DETAILED PREDICTIONS ON TEST SET (First 20 claims)")
print("=" * 80)

results_dict = {'Actual': y_test_original}
results_dict.update({name: models_pred[name] for name in model_list})
results_df = pd.DataFrame(results_dict).reset_index(drop=True)

print(f"\nShowing actual claim values vs predictions from all {len(model_list)} models (in EUR):\n")
display(results_df.head(20).round(2))

print("\n" + "=" * 80)
print("MODEL PREDICTION ERRORS (First 20 claims)")
print("=" * 80)
print("\nAbsolute Error (EUR) for each model:\n")

errors_dict = {'Actual': y_test_original}
errors_dict.update({f'{name} Error': np.abs(models_pred[name] - y_test_original) for name in model_list})
errors_df = pd.DataFrame(errors_dict).reset_index(drop=True)
display(errors_df.head(20).round(2))

print("\n" + "=" * 80)
print("MODEL ERROR STATISTICS")
print("=" * 80)

error_stats = pd.DataFrame({
    'Model': model_list,
    'Mean Absolute Error (EUR)': [np.abs(models_pred[name] - y_test_original).mean() for name in model_list],
    'Median Absolute Error (EUR)': [np.median(np.abs(models_pred[name] - y_test_original)) for name in model_list],
    'Max Error (EUR)': [np.abs(models_pred[name] - y_test_original).max() for name in model_list]
})

print("\n")
display(error_stats.round(2))

print("\n" + "=" * 80)
print("TO VIEW MLFLOW DASHBOARD:")
print("   Run in terminal: mlflow ui")
print("   Then open: http://localhost:5000")
print("=" * 80)


MODEL COMPARISON: ALL 5 MODELS

Note: Detailed metrics, hyperparameters, and model artifacts are now
tracked in MLflow. View the dashboard with: mlflow ui

Model Performance (RMSE on original scale, EUR):
  1. Linear Regression:              €24445.42
  2. XGBoost (basic):                €12301.53
  3. Random Forest:                  €12309.59
  4. XGBoost (tuned, log-normal):    €12336.02
  5. Actuarial XGBoost (gamma):      €12113.84

Model Performance (MAPE):
  1. Linear Regression:              59.66%
  2. XGBoost (basic):                40.94%
  3. Random Forest:                  40.20%
  4. XGBoost (tuned, log-normal):    40.62%
  5. Actuarial XGBoost (gamma):      50.69%

BEST MODEL SUMMARY

Best Model: Actuarial XGBoost
  RMSE: €12113.84
  MAPE: 50.69%
  Average Claim Value: €80826.68

DETAILED PREDICTIONS ON TEST SET (First 20 claims)

Showing actual claim values vs predictions from all 7 models (in EUR):



,Actual,Linear Regression,XGBoost (Basic),Random Forest,XGBoost (Tuned),Actuarial XGBoost
0,27276.0,20660.90,14925.750000,14616.00,14702.480469,20657.580078
1,142504.0,173081.01,137813.781250,138646.77,136330.343750,136552.937500
2,119619.0,85110.61,101554.562500,102209.49,100336.656250,101294.882812
3,157104.0,177725.85,138285.625000,139011.01,138559.421875,138352.453125
4,86368.0,84504.83,100264.539062,99866.76,100155.492188,100456.203125
5,25232.0,22415.20,17802.400391,18232.49,17450.039062,22141.519531
6,70311.0,42082.29,59659.910156,59920.82,60059.460938,61711.738281
7,147760.0,173552.66,139915.703125,141316.52,141610.234375,140691.562500
8,126357.0,175923.04,138440.546875,138138.61,140232.031250,138823.234375
9,73317.0,43572.23,63417.281250,59866.29,62504.558594,63411.390625



MODEL PREDICTION ERRORS (First 20 claims)

Absolute Error (EUR) for each model:



,Actual,Linear Regression Error,XGBoost (Basic) Error,Random Forest Error,XGBoost (Tuned) Error,Actuarial XGBoost Error
0,27276.0,6615.10,12350.25,12660.00,12573.52,6618.42
1,142504.0,30577.01,4690.22,3857.23,6173.66,5951.06
2,119619.0,34508.39,18064.44,17409.51,19282.34,18324.12
3,157104.0,20621.85,18818.38,18092.99,18544.58,18751.55
4,86368.0,1863.17,13896.54,13498.76,13787.49,14088.20
5,25232.0,2816.80,7429.60,6999.51,7781.96,3090.48
6,70311.0,28228.71,10651.09,10390.18,10251.54,8599.25
7,147760.0,25792.66,7844.30,6443.48,6149.77,7068.44
8,126357.0,49566.04,12083.55,11781.61,13875.03,12466.23
9,73317.0,29744.77,9899.71,13450.71,10812.45,9905.61



MODEL ERROR STATISTICS




,Model,Mean Absolute Error (EUR),Median Absolute Error (EUR),Max Error (EUR)
0,Linear Regression,19792.70,17593.86,140150.20
1,XGBoost (Basic),9883.20,9447.66,145093.59
2,Random Forest,9855.08,9508.80,144340.07
3,XGBoost (Tuned),9910.33,9538.71,145330.92
4,Actuarial XGBoost,9739.74,9439.67,140867.57



TO VIEW MLFLOW DASHBOARD:
   Run in terminal: mlflow ui
   Then open: http://localhost:5000


## 6. Save & Track the Best Model

### Three Ways Your Model Is Saved

**1. MLflow (Primary Storage)**
- **What**: All 7 models automatically saved by MLflow's autologging
- **Where**: `./mlruns/0/[RUN_ID]/artifacts/model/`
- **Why**: Official experiment record, reproducible, tracked metadata
- **How to access**: Open MLflow dashboard (`mlflow ui`), find your run, download from Artifacts
- **Best for**: Production deployment, team sharing, compliance

**2. Local Backup Directory**
- **What**: Best model saved as `.pkl` file
- **Where**: `./best_model_backup/best_model.pkl`
- **Why**: Quick local reference, fast to load for inference
- **How to access**: `joblib.load('./best_model_backup/best_model.pkl')`
- **Best for**: Quick testing, local development

**3. Hugging Face Hub (Cloud)**
- **What**: Model uploaded to Hugging Face's free model repository
- **Where**: `https://huggingface.co/yourname/belgian-mtpl-claim-severity`
- **Why**: Free cloud storage, accessible from anywhere, easy to integrate with Streamlit
- **How to access**: `hf_hub_download(repo_id="username/repo", filename="model.pkl")`
- **Best for**: Streamlit apps, sharing with non-technical users, public models

### What Gets Saved?
- **model.pkl**: The trained machine learning model
- **preprocessor.pkl**: The data transformation pipeline (StandardScaler + OneHotEncoder)
- **metadata.json**: Model info (type, performance metrics, features used)
- **BEST_MODEL_INFO.txt**: Human-readable summary document
- **README.md**: Instructions for loading and using the model

### The Workflow
1. Train all 7 models (each automatically logged to MLflow)
2. This cell identifies which one performed best
3. Best model automatically saved to local backup
4. You can then upload to Hugging Face Hub (optional, Section 7)
5. Deploy to production via your chosen method



In [ ]:
import joblib
import os

print("\n" + "=" * 80)
print("BEST MODEL TRACKING & ARTIFACT MANAGEMENT")
print("=" * 80)

print(f"\nBest Model: {best_model}")
print(f"  RMSE (EUR): €{models_rmse[best_model]:.2f}")
print(f"  MAPE: {models_mape[best_model] * 100:.2f}%")

print("\n" + "-" * 80)
print("MLflow Artifact Storage (Primary)")
print("-" * 80)
print("""
All trained models are automatically saved as MLflow artifacts:
Location: ./mlruns/0/[RUN_ID]/artifacts/model/

To access the best model:
1. Open MLflow Dashboard: mlflow ui
2. Go to http://localhost:5000
3. Find the run: "XGB_Tuned_RandomSearch" (or whichever model is best)
4. Download the model from the Artifacts section
5. Load it: mlflow.pyfunc.load_model('path/to/artifacts/model')

Or programmatically:
""")

print("Example code to load best model from MLflow:")
print("""
import mlflow
from mlflow import MlflowClient

client = MlflowClient()
runs = client.search_runs(experiment_ids=['0'], order_by=['metrics.rmse_eur ASC'], max_results=1)
best_run = runs[0]
model_uri = f"runs:/{best_run.info.run_id}/model"
best_model = mlflow.sklearn.load_model(model_uri)
best_model_pyfunc = mlflow.pyfunc.load_model(model_uri)
""")

print("\n" + "-" * 80)
print("Local Backup (Optional)")
print("-" * 80)

backup_dir = "./best_model_backup"
if not os.path.exists(backup_dir):
    os.makedirs(backup_dir)

model_mapping = {
    'Linear Regression': lambda: model_lr,
    'XGBoost (basic)': lambda: model_xgb,
    'Random Forest': lambda: model_rf,
    'XGBoost (tuned)': lambda: best_model_xgb,
    'Actuarial XGBoost': lambda: model_xgb_actuarial,
    'Ridge Regression': lambda: model_ridge,
    'Lasso Regression': lambda: model_lasso
}

model_objects = {}
for name, get_model in model_mapping.items():
    try:
        model_objects[name] = get_model()
    except NameError:
        pass

if best_model not in model_objects:
    print(f"Warning: Best model '{best_model}' not found in model_objects dictionary.")
    print("   This may happen if the model wasn't trained yet.")
    print("   Please run the corresponding training cell first.")
else:
    best_model_object = model_objects[best_model]
    backup_model_path = os.path.join(backup_dir, "best_model.pkl")
    joblib.dump(best_model_object, backup_model_path)
    print(f"Local backup saved to: {backup_model_path}")

    if best_model == 'Actuarial XGBoost':
        backup_preprocessor_path = os.path.join(backup_dir, "preprocessor.pkl")
        joblib.dump(preprocessor_actuarial, backup_preprocessor_path)
    else:
        backup_preprocessor_path = os.path.join(backup_dir, "preprocessor.pkl")
        joblib.dump(preprocessor, backup_preprocessor_path)

    print(f"Preprocessor backup saved to: {backup_preprocessor_path}")

    summary_path = os.path.join(backup_dir, "BEST_MODEL_INFO.txt")
    with open(summary_path, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("BEST MODEL INFORMATION\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"Model: {best_model}\n")
        f.write(f"RMSE (EUR): €{models_rmse[best_model]:.2f}\n")
        f.write(f"MAPE: {models_mape[best_model] * 100:.2f}%\n")
        f.write(f"Average Claim Value: €{y_test_original.mean():.2f}\n\n")
        f.write("MASTER SOURCE: MLflow\n")
        f.write("- All models and artifacts are stored in: ./mlruns/\n")
        f.write("- View via MLflow Dashboard: mlflow ui\n")
        f.write("- Load via MLflow API for production use\n\n")
        f.write("LOCAL BACKUP: ./best_model_backup/\n")
        f.write("- best_model.pkl: The trained model\n")
        f.write("- preprocessor.pkl: Data preprocessing pipeline\n")
        f.write("- This is for quick local development reference\n\n")
        f.write("FEATURES USED:\n")
        for i, feature in enumerate(features, 1):
            f.write(f"  {i}. {feature}\n")

    print(f"Summary saved to: {summary_path}")

print("\n" + "=" * 80)
print("RECOMMENDATION: Use MLflow for production deployment")
print("The ./mlruns/ directory is the authoritative source for all models")
print("=" * 80)


## 7. Deploy to Cloud: Hugging Face Hub (Optional)

### What Is Hugging Face Hub?
Hugging Face is an open-source community platform where people share machine learning models. It's free to upload models and provides:
- **Cloud storage**: Your model is hosted online
- **Easy access**: Load models from anywhere with one line of code
- **Version control**: Track changes to your model
- **Community visibility**: Share your work with other ML enthusiasts

### When to Use This

✅ **Use Hugging Face Hub if you want to**:
- Build a Streamlit web app
- Share models with team members
- Access models from different computers
- Keep models accessible for months/years
- Show off your work publicly

❌ **Skip this if you only need**:
- Local development
- Testing models on your computer
- Private internal use (use MLflow instead)

### How It Works
1. Create a free account at https://huggingface.co/
2. Generate an API token
3. Set the token in your `.env` file as `HF_TOKEN=...`
4. Set `UPLOAD_TO_HUB = True` in the cell below
5. Run the cell to upload your model
6. Your model is now accessible from anywhere!

### Using Models from Hugging Face in Streamlit
Once your model is uploaded, your Streamlit app can load it:
```python
from huggingface_hub import hf_hub_download
import joblib

model = joblib.load(
    hf_hub_download(
        repo_id="your_username/belgian-mtpl-claim-severity",
        filename="model.pkl"
    )
)
```

No need to store model files in your GitHub repository - they're in the cloud!



In [ ]:
%pip install huggingface_hub -q
import os
import json
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

UPLOAD_TO_HUB = True
HF_TOKEN = os.getenv("HF_TOKEN")
HF_REPO_NAME = "belgian-mtpl-claim-severity"

print("\n" + "=" * 80)
print("PUSH MODEL TO HUGGING FACE HUB (FREE CLOUD STORAGE)")
print("=" * 80)

if UPLOAD_TO_HUB:
    try:
        from huggingface_hub import HfApi, login
        import shutil
        
        if HF_TOKEN is None:
            HF_TOKEN = os.getenv("HF_TOKEN")
        
        if HF_TOKEN is None:
            print("ERROR: HF_TOKEN not found!")
            print("Set HF_TOKEN environment variable or pass it directly")
        else:
            login(token=HF_TOKEN)
            
            temp_dir = "./hf_upload_temp"
            if os.path.exists(temp_dir):
                shutil.rmtree(temp_dir)
            os.makedirs(temp_dir)
            
            model_mapping = {
                'Linear Regression': lambda: model_lr,
                'XGBoost (basic)': lambda: model_xgb,
                'Random Forest': lambda: model_rf,
                'XGBoost (tuned)': lambda: best_model_xgb,
                'Actuarial XGBoost': lambda: model_xgb_actuarial,
                'Ridge Regression': lambda: model_ridge,
                'Lasso Regression': lambda: model_lasso
            }
            
            model_objects = {}
            for name, get_model in model_mapping.items():
                try:
                    model_objects[name] = get_model()
                except NameError:
                    pass
            
            if best_model not in model_objects:
                print(f"Error: Best model '{best_model}' not found in model_objects dictionary.")
                print("   Please run the corresponding training cell first.")
            else:
                best_model_object = model_objects[best_model]
                
                joblib.dump(best_model_object, os.path.join(temp_dir, "model.pkl"))
                
                if best_model == 'Actuarial XGBoost':
                    joblib.dump(preprocessor_actuarial, os.path.join(temp_dir, "preprocessor.pkl"))
                else:
                    joblib.dump(preprocessor, os.path.join(temp_dir, "preprocessor.pkl"))
                
                metadata = {
                    'model_name': best_model,
                    'rmse_eur': float(models_rmse[best_model]),
                    'mape': float(models_mape[best_model]),
                    'features': features,
                    'uploaded_date': datetime.now().isoformat(),
                    'target': 'claim_value (EUR)',
                    'data_source': 'Belgian Motor Third-Party Liability (beMTPL16)'
                }
                
                with open(os.path.join(temp_dir, "metadata.json"), 'w') as f:
                    json.dump(metadata, f, indent=2)
            
            print(f"\nUploading model to Hugging Face Hub...")
            print(f"Repository: {HF_REPO_NAME}")
            
            api = HfApi()
            api.create_repo(repo_id=HF_REPO_NAME, private=False, exist_ok=True)
            api.upload_folder(
                folder_path=temp_dir,
                repo_id=HF_REPO_NAME,
                commit_message=f"Upload best model: {best_model} with RMSE €{models_rmse[best_model]:.2f}"
            )
            
            print(f"\nSuccess! Model uploaded to Hugging Face Hub")
            print(f"Repository URL: https://huggingface.co/{api.whoami()['name']}/{HF_REPO_NAME}")
            print(f"\nYou can now use this in Streamlit:")
            print(f"  from huggingface_hub import hf_hub_download")
            print(f"  model = joblib.load(hf_hub_download(repo_id='YOUR_USERNAME/{HF_REPO_NAME}', filename='model.pkl'))")
            
            shutil.rmtree(temp_dir)
            
    except ImportError:
        print("ERROR: huggingface_hub package not installed")
        print("Install with: pip install huggingface_hub")
else:
    print("\nHugging Face Hub Upload is DISABLED")
    print("\nTo enable:")
    print("  1. Create free account at https://huggingface.co/")
    print("  2. Generate token at https://huggingface.co/settings/tokens")
    print("  3. Set HF_TOKEN environment variable:")
    print("     export HF_TOKEN='your_token_here'")
    print("  4. Set UPLOAD_TO_HUB = True above")
    print("  5. Run this cell")
    print("\nOr pass token directly: HF_TOKEN = 'your_token_here'")


## Summary & Next Steps

### What You've Built
Congratulations! You've created a complete machine learning pipeline for insurance claim severity prediction. Here's what you accomplished:

✅ **Data Pipeline**
- Loaded and validated 70,791 insurance claims
- Handled missing values intelligently (smart imputation)
- Removed problematic columns to prevent data leakage
- Engineered 10 predictive features

✅ **Model Comparison**
- Trained 7 different machine learning models
- Linear Regression (baseline)
- XGBoost (basic and tuned versions)
- Random Forest (ensemble)
- Actuarial XGBoost (specialized for insurance)
- Ridge Regression (with L2 regularization)
- Lasso Regression (with L1 regularization)

✅ **Experiment Tracking**
- Tracked all models using MLflow
- Recorded hyperparameters, metrics, and artifacts
- Can reproduce results and compare models anytime

✅ **Model Persistence**
- Saved best model to local disk
- Optionally uploaded to Hugging Face Hub
- Created documentation and metadata

### Next Steps: From Research to Production

**Step 1: Evaluate & Validate**
- Review the best model's performance
- Check for any concerns or biases
- Decide if it's ready for business use

**Step 2: Build a Web App (Optional)**
- Use Streamlit to create an interactive app
- Load model from Hugging Face Hub
- Let insurance agents make predictions on new claims

**Step 3: Deploy to Production**
- Choose your deployment platform:
  - **Streamlit Cloud**: Free, easy to use
  - **Docker + Cloud Run**: Scalable
  - **AWS/Azure/GCP**: Enterprise-grade
- Set up monitoring to track model performance
- Create a retraining schedule

**Step 4: Monitor & Maintain**
- Track prediction accuracy over time
- Retrain model monthly/quarterly with new data
- Update features if business logic changes

### Key Files & Locations

| Item | Location | Purpose |
|------|----------|---------|
| **Experiment history** | `./mlruns/` | MLflow tracking database |
| **Best model (local)** | `./best_model_backup/best_model.pkl` | Quick development reference |
| **Best model (cloud)** | Hugging Face Hub | Production access |
| **Configuration** | `.env` | Secrets (HF_TOKEN) |
| **.gitignore** | `.gitignore` | Prevents secrets from being pushed to GitHub |

### Important Commands

```bash
# View MLflow dashboard
mlflow ui

# Then open browser to: http://localhost:5000
```

### Troubleshooting

**Problem**: Model accuracy is low
- Review feature engineering choices
- Check for data quality issues
- Try different models or parameters

**Problem**: Model takes too long to train
- Reduce dataset size for testing
- Use `n_jobs=-1` for parallel processing
- Try simpler models (Linear Regression)

**Problem**: Predictions don't make business sense
- Check preprocessing pipeline
- Verify model is predicting in EUR
- Review feature scaling

---

**Ready to deploy?** Start with Section 7 to upload your model to Hugging Face Hub, then build your Streamlit app!